# Process Sentinel-1 partial products

This tutorial processes `.partial.SAFE` products created by `tutorial-s1-partial-product-download.ipynb`.

Key behavior to remember:

- A partial product carries its own `partial_aoi.geojson`.
- User-facing processors use this stored AOI and ignore the `shp` argument for valid partial products.
- The `shp` argument is still present for API compatibility with regular products.
- InSAR partial processing currently requires both products to be partial products with identical stored AOIs and matching downloaded IW/polarization subsets.
- Burst overlap and burst-index offsets are checked later in preprocessing.

## 1. Imports and paths

In [ ]:
import logging
from math import pi
from pathlib import Path

import folium
import geopandas as gpd
from folium import LayerControl

from eo_tools.S1.core import read_partial_aoi, read_partial_download_info
from eo_tools.S1.process import process_h_alpha_dual, process_insar, process_slc
from eo_tools_dev.util import palette_phi, serve_map, show_cog

logging.basicConfig(level=logging.INFO)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
data_dir = Path("/data/S1/partial_dls")
output_root = Path("/data/res/tutorial-s1-partial-products")
output_root.mkdir(parents=True, exist_ok=True)

ids = [
    "S1A_IW_SLC__1SDV_20230904T063730_20230904T063757_050174_0609E3_DAA1",
    "S1A_IW_SLC__1SDV_20230916T063730_20230916T063757_050349_060FCD_6814",
]

primary_path = data_dir / f"{ids[0]}.partial.SAFE"
secondary_path = data_dir / f"{ids[1]}.partial.SAFE"

for path in (primary_path, secondary_path):
    if not path.is_dir():
        raise FileNotFoundError(
            f"Missing partial product: {path}. Run the partial-product download tutorial first."
        )

primary_path, secondary_path

## 2. Inspect the partial metadata

The processor reads this metadata when creating product objects. It uses the stored AOI for high-level processing, and uses the manifest to validate requested subswaths, polarizations, and available bursts.

In [ ]:
primary_manifest = read_partial_download_info(primary_path)
secondary_manifest = read_partial_download_info(secondary_path)
primary_aoi = read_partial_aoi(primary_path, primary_manifest)
secondary_aoi = read_partial_aoi(secondary_path, secondary_manifest)

print(primary_manifest)
print(f"Primary and secondary stored AOIs are equal: {primary_aoi.equals_exact(secondary_aoi, 0)}")

The next cell defines `shp` from the original tutorial AOI only to keep the processor calls compatible with regular-product examples. For these partial products, the high-level processors will use `partial_aoi.geojson` from the product folders instead.

In [ ]:
aoi_file = Path("/eo_tools/data/Morocco_tiny.geojson")
shp = gpd.read_file(aoi_file).geometry.iloc[0]

## 3. Process an InSAR pair

For the initial partial-product release, InSAR processing is intentionally strict: both inputs must be partial products with equal stored AOIs and matching downloaded IW/polarization subsets. If the secondary product cannot provide the offset-mapped bursts needed by the primary-selected range, preprocessing raises a clear error.

In [ ]:
insar_dir = process_insar(
    prm_path=str(primary_path),
    sec_path=str(secondary_path),
    output_dir=str(output_root / "insar"),
    aoi_name=None,
    shp=shp,
    pol="vv",
    subswaths=["IW1", "IW2", "IW3"],
    write_coherence=True,
    write_interferogram=True,
    write_primary_amplitude=False,
    write_secondary_amplitude=False,
    apply_fast_esd=True,
    dem_upsampling=1.8,
    dem_force_download=False,
    dem_buffer_arc_sec=40,
    boxcar_coherence=[3, 3],
    filter_ifg=True,
    multilook=[1, 4],
    warp_kernel="bicubic",
    cal_type="beta",
    clip_to_shape=True,
    orb_dir="/data/S1_orbits/",
)

insar_dir

In [ ]:
m = folium.Map()
show_cog(f"{insar_dir}/phi_vv.tif", m, rescale=f"{-pi},{pi}", colormap=palette_phi())
show_cog(f"{insar_dir}/coh_vv.tif", m, rescale="0,1")
LayerControl().add_to(m)
serve_map(m)

## 4. Process one partial SLC product

`process_slc(...)` follows the single-product partial contract: the stored AOI is used, and the requested polarization/subswath must exist in `partial_download.yml`.

In [ ]:
slc_dir = process_slc(
    slc_path=str(primary_path),
    output_dir=str(output_root / "slc-beta"),
    aoi_name=None,
    shp=shp,
    pol="full",
    subswaths=["IW1", "IW2", "IW3"],
    dem_upsampling=1.8,
    dem_force_download=False,
    dem_buffer_arc_sec=40,
    multilook=[1, 4],
    warp_kernel="bicubic",
    cal_type="beta",
    clip_to_shape=True,
    orb_dir="/data/S1_orbits/",
)

slc_dir

In [ ]:
m = folium.Map()
show_cog(f"{slc_dir}/amp_vv.tif", m, rescale="0,1")
show_cog(f"{slc_dir}/amp_vh.tif", m, rescale="0,1")
LayerControl().add_to(m)
serve_map(m)

## 5. Optional: dual-pol H-alpha processing

This requires the partial product to contain both `VV` and `VH`. If the product was downloaded with `pol="vv"` only, this processor should raise an explicit partial-manifest validation error.

In [ ]:
h_alpha_dir = process_h_alpha_dual(
    slc_path=str(primary_path),
    output_dir=str(output_root / "h-alpha"),
    aoi_name=None,
    shp=shp,
    subswaths=["IW1", "IW2", "IW3"],
    dem_upsampling=1.8,
    dem_force_download=False,
    dem_buffer_arc_sec=40,
    multilook=[1, 4],
    warp_kernel="bicubic",
    cal_type="beta",
    clip_to_shape=True,
    orb_dir="/data/S1_orbits/",
)

h_alpha_dir

In [ ]:
m = folium.Map()
show_cog(f"{h_alpha_dir}/amp_vv.tif", m, rescale="0,1")
show_cog(f"{h_alpha_dir}/amp_vh.tif", m, rescale="0,1")
show_cog(f"{h_alpha_dir}/alpha.tif", m, rescale="0,90")
show_cog(f"{h_alpha_dir}/H.tif", m, rescale="0,1")
LayerControl().add_to(m)
serve_map(m)

## Troubleshooting

- `Missing partial product`: run the download tutorial first and check `data_dir`.
- `requested polarization/subswath is not available`: re-download the partial product with the required `pol` selection.
- `secondary partial product does not contain required bursts`: the primary AOI-selected bursts do not all have offset-mapped secondary data available; re-download a compatible pair or use a smaller AOI.
- unexpected clipping or geocoding extent: inspect `partial_aoi.geojson`; this AOI is the processing reference for partial products.